<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-3-rag-pipelines/Module_3_Session_1_Embeddings_and_Similarity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3 - Session 1: Embeddings & Similarity by Hand
Goal: Turn Swiggy FAQ questions into number-vectors and find the closest match by hand.

In [ ]:
# Install sentence-transformers.
# This library downloads a small pre-trained model that converts
# any sentence into an "embedding" — a list of numbers capturing meaning.
!pip install -q sentence-transformers

## Step 2 - Load the Embedding Model
We use a small pre-trained model called all-MiniLM-L6-v2.
It converts any sentence into a vector of 384 numbers.

In [ ]:
# Import the SentenceTransformer class from the library we just installed
# SentenceTransformer is the main tool — give it a model name, it downloads
# and loads that model for us
from sentence_transformers import SentenceTransformer

# Load a small but powerful model called all-MiniLM-L6-v2
# "MiniLM" means it is a lightweight model — fast and free to run on Colab
# It converts any sentence into exactly 384 numbers (384-dimensional vector)
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully!")
print(f"This model outputs vectors of size: {model.get_sentence_embedding_dimension()}")

## Step 3 - Our Swiggy FAQ Questions
These are the questions stored in our knowledge base.
We will convert each one into a vector (embedding).

In [ ]:
# These are Swiggy FAQ questions — our "knowledge base"
# In a real system these would be thousands of help articles
# For today we use 6 questions to learn the concept clearly
swiggy_faq_questions = [
    "Where is my order?",
    "How do I cancel my order?",
    "My food arrived cold, what should I do?",
    "How do I apply a coupon code?",
    "My payment failed but money was deducted.",
    "How do I change my delivery address?"
]

print(f"We have {len(swiggy_faq_questions)} FAQ questions in our knowledge base")

## Step 4 - Convert FAQ Questions into Embeddings
We pass all 6 questions to the model at once.
Each question becomes a vector of 384 numbers.

In [ ]:
# .encode() is the main method of SentenceTransformer
# You pass it a list of sentences, it returns a list of vectors
# Each vector is a numpy array of 384 numbers
faq_embeddings = model.encode(swiggy_faq_questions)

# Let's see the shape — should be (6, 384)
# 6 questions, each converted to 384 numbers
print(f"Shape of embeddings: {faq_embeddings.shape}")

# Let's peek at the first 5 numbers of the first question's vector
# just to see what these numbers actually look like
print(f"\nFirst question: '{swiggy_faq_questions[0]}'")
print(f"First 5 numbers of its vector: {faq_embeddings[0][:5]}")

## Step 5 - Embed the User's Query
The customer's question also becomes a vector.
We will then compare it against all 6 FAQ vectors.

In [ ]:
# This is what the Swiggy customer typed
# Notice it is NOT exactly any of our FAQ questions — different words, same meaning
user_query = "I can't find my order, it's been 30 minutes"

# Convert the user query into a vector — same model, same 384 dimensions
query_embedding = model.encode(user_query)

print(f"User query: '{user_query}'")
print(f"Query vector shape: {query_embedding.shape}")
print(f"First 5 numbers: {query_embedding[:5]}")

## Step 6 - Cosine Similarity
Cosine similarity gives a score between 0 and 1.
1 means identical meaning. 0 means completely unrelated.
We compare the user query against all 6 FAQ questions.

In [ ]:
# util is a helper module inside sentence_transformers
# pytorch_cos_sim computes cosine similarity between two sets of vectors
# It returns a score between 0 and 1 for each FAQ question
from sentence_transformers import util

# Compare the query vector against all 6 FAQ vectors at once
scores = util.pytorch_cos_sim(query_embedding, faq_embeddings)

# Print each FAQ question with its similarity score
print(f"Query: '{user_query}'\n")
print("Similarity scores against each FAQ question:")
print("-" * 55)

for i, question in enumerate(swiggy_faq_questions):
    print(f"{scores[0][i]:.4f}  →  {question}")

## Step 7 - Automatically Find the Best Matching FAQ
Instead of reading scores manually, we find the highest score programmatically.

In [ ]:
# .argmax() finds the index of the highest score
# That index tells us which FAQ question is the best match
best_match_index = scores.argmax()

print(f"User query: '{user_query}'")
print(f"\nBest matching FAQ: '{swiggy_faq_questions[best_match_index]}'")
print(f"Similarity score: {scores[0][best_match_index]:.4f}")

## Step 8 - Send Retrieved Context to LLM
We pass the user query + best matching FAQ to the LLM.
The LLM uses the retrieved context to generate a grounded reply.
This is the core idea of RAG — Retrieve, then Generate.

In [ ]:
!pip install groq
!python -c "import groq; print('groq version:', groq.__version__)"

In [ ]:
import os
from groq import Groq
from google.colab import userdata

# Load Groq API key from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq()

# The retrieved FAQ question is our context
retrieved_context = swiggy_faq_questions[best_match_index]

# Build the prompt — user query + retrieved context together
prompt = f"""You are a Swiggy customer support agent.

Use the following FAQ context to answer the customer's question.
Only use the context provided — do not make up information.

FAQ Context: {retrieved_context}

Customer Question: {user_query}

Reply in a friendly, helpful tone in 2-3 sentences."""

# Send to LLM
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)

print(f"Customer asked: '{user_query}'")
print(f"\nRetrieved FAQ: '{retrieved_context}'")
print(f"\nSwiggy Support Reply:")
print(response.choices[0].message.content)

## What We Built
- Converted Swiggy FAQ questions into embedding vectors using sentence-transformers
- Converted a customer query into a vector using the same model
- Computed cosine similarity to find the most relevant FAQ
- Sent the retrieved FAQ + customer query to Groq LLM
- Got a grounded support reply — no hallucination, context-based answer

## AWS Equivalent
| What we did here         | AWS service                          |
|--------------------------|--------------------------------------|
| sentence-transformers    | Amazon Titan Embeddings              |
| Cosine similarity search | Amazon OpenSearch Serverless         |
| Groq LLM                 | Amazon Bedrock (Claude/Llama)        |
| Full RAG pipeline        | Amazon Bedrock Knowledge Bases       |

## What is Next
Session 2 — store thousands of embeddings in a real vector database (FAISS)
and query them in milliseconds instead of looping by hand.